# ODRD — coleta retrospectiva e conjunto de calibração (v2.1)

**Por quê.** A cota da API ficou estourada em 45 de 60 dias, entre 25/08 e 13/10/2026. Os 449 alertas que saíram do painel sem vídeo estão contados como silêncio **sem prova de que a busca rodou**. O primeiro lote mostrou ainda que os filtros de relevância nunca foram medidos contra julgamento humano. Por isso este notebook passou a ser também o **coletor do estudo de calibração**: ele guarda *todos* os candidatos da busca, com os atributos usados na decisão.

**Fases** (parâmetro `FASE`):
1. `calibracao+amostra`: 38 alertas com vídeos conhecidos e 120 alertas de silêncio (amostra sistemática por UF × mês).
2. `variantes`: nos 38 alertas de calibração, roda 3 consultas alternativas (sem UF; "chuva + Município"; município entre aspas). O resultado de todas é julgado junto (*pooling*) para medir a recuperação de cada consulta.
3. `tudo`: os 329 alertas de silêncio restantes. Só depois de congelados os filtros calibrados (v0.4).

**Novidades da v2 (pipeline 0.3.3):**
- `videos.csv` guarda canal, descrição, tags e a decisão de cada candidato.
- Os 12 alertas antigos sem tipo de evento foram corrigidos e são refeitos automaticamente.
- Comentários sem autor e com as menções a outros usuários trocadas por `@usuario`.
- A célula **Complemento** busca a descrição dos vídeos do lote 1 (1 unidade a cada 50 vídeos, sem gastar buscas).
- **v2.1:**
  - respeita o limite de 100 buscas *por minuto* (intervalo de 0,8 s e nova tentativa após 65 s em caso de erro 429);
  - a célula **Limpeza** separa os comentários de coletas substituídas.

**Como ler o resultado** (Efstratiou, 2025): achar um vídeo que cita o município **refuta** o silêncio; não achar é **inconclusivo**. Os critérios de aceite aqui são os da versão atual do pipeline, que ainda não foram calibrados: a decisão final virá da anotação humana.

**Registro.** Instrumento separado do pipeline, com projeto do Google Cloud e chave próprios. Descreva-o assim no protocolo e no CEP. Textos ficam só no Drive.

**Uso.** Rode tudo uma vez por dia. O notebook para sozinho no limite de buscas e retoma de onde parou.

In [ ]:
# ===== Parâmetros =====
FASE = "calibracao+amostra"     # "calibracao+amostra" | "variantes" | "tudo"
MAX_BUSCAS_EXECUCAO = 95         # search.list: limite de 100/dia por projeto
MAX_UNIDADES_EXECUCAO = 9000     # demais chamadas: limite de 10.000/dia
COLETAR_COMENTARIOS = True       # janela de 72 h, só vídeos que citam o município, só no grupo silêncio
MAX_PAGINAS_COMENTARIOS = 50     # 100 tópicos por página
PASTA = "/content/drive/MyDrive/ODRD/retro_backfill"
BRANCH = "main"

In [ ]:
# ===== Preparação =====
import csv, os, shutil, subprocess, sys, time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
REPO = "/content/crisis-dashboard"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/alvarocoutinho/crisis-dashboard.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
if REPO not in sys.path:
    sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m == "faa_poc" or m.startswith("faa_poc.")]:
    del sys.modules[m]   # recarrega o código atualizado sem reiniciar o ambiente

from faa_poc import youtube
from faa_poc.config import CONFIG
from faa_poc.constants import FILTER_VERSION, YT_SEARCH_TERMS
from faa_poc.evidence import is_municipal, municipal_evidence
from faa_poc.geo import score_video
from faa_poc.text import pseudonymize_mentions
from faa_poc.timeutils import parse_dt

CFG = {**CONFIG, "MAX_VIDEOS_PER_ALERT": 10}   # igual ao pipeline (scripts/pipeline.py)
KEY = userdata.get("YT_key")
assert KEY, "Segredo 'YT_key' vazio: confira o ícone de chave na barra lateral e o acesso do notebook."

pasta = Path(PASTA); pasta.mkdir(parents=True, exist_ok=True)
ARQ_RES, ARQ_VID, ARQ_COM = pasta / "resultados_alertas.csv", pasta / "videos.csv", pasta / "comentarios.csv"
ARQ_VAR, ARQ_COMPL = pasta / "resultados_variantes.csv", pasta / "videos_complemento.csv"
print(f"código do pipeline: {COMMIT} | filtros {FILTER_VERSION} | saída: {pasta}")

In [ ]:
# ===== Migração do lote 1 (roda uma vez; idempotente) =====
# Comentários: troca menções a usuários por @usuario (cópia do original fica só até você apagar)
if ARQ_COM.exists():
    with open(ARQ_COM, encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    if rows and any(r.get("text_clean", "") != pseudonymize_mentions(r.get("text_clean", "")) for r in rows):
        shutil.copy(ARQ_COM, pasta / "comentarios_lote1_ORIGINAL_apagar.csv")
        for r in rows:
            r["text_clean"] = pseudonymize_mentions(r.get("text_clean", ""))
        with open(ARQ_COM, "w", encoding="utf-8", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
        print(f"{len(rows)} comentários pseudonimizados. Confira e APAGUE comentarios_lote1_ORIGINAL_apagar.csv do Drive.")
    else:
        print("comentários já pseudonimizados")

In [ ]:
# ===== Universo e checkpoint =====
with open(f"{REPO}/data/retro/universo_alertas.csv", encoding="utf-8") as fh:
    universo = list(csv.DictReader(fh))

def _ler(arq):
    if not arq.exists():
        return []
    with open(arq, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

ev_universo = {r["alert_id"]: r["evento"] for r in universo}
# feito = coletado com sucesso E com o mesmo tipo de evento do universo atual
# (os 12 alertas antigos sem tipo são refeitos com as consultas certas)
feitos = {r["alert_id"] for r in _ler(ARQ_RES)
          if r["status"] == "ok" and r.get("evento", "") == ev_universo.get(r["alert_id"], "")}
var_feitas = {(r["alert_id"], r["variante"]) for r in _ler(ARQ_VAR) if r["status"] == "ok"}
filtro = {
    "calibracao+amostra": lambda r: r["grupo"] == "calibracao" or r["amostra"] == "1",
    "variantes": lambda r: r["grupo"] == "calibracao",
    "tudo": lambda r: True,
}[FASE]
VARIANTES = ("sem_uf", "chuva", "aspas")
if FASE == "variantes":
    pendentes = [(r, v) for r in sorted(universo, key=lambda r: int(r["ordem"])) if filtro(r)
                 for v in VARIANTES if (r["alert_id"], v) not in var_feitas]
    print(f"{len(pendentes)} buscas de variante pendentes (~{len(pendentes) / MAX_BUSCAS_EXECUCAO:.1f} execuções)")
else:
    pendentes = [(r, None) for r in sorted(universo, key=lambda r: int(r["ordem"]))
                 if filtro(r) and r["alert_id"] not in feitos]
    print(f"{len(universo)} alertas; {len(feitos)} já coletados; {len(pendentes)} pendentes na fase '{FASE}' "
          f"(~{len(pendentes) * 2 / MAX_BUSCAS_EXECUCAO:.1f} execuções)")

In [ ]:
# ===== Coleta =====
CAMPOS_RES = ["alert_id", "grupo", "amostra", "ordem", "municipio", "uf", "evento", "opened_at",
              "coletado_em", "codigo_pipeline", "filter_version", "busca_de", "busca_ate", "consultas", "status", "erro",
              "n_candidatos", "n_aceitos", "n_aceitos_municipais", "n_candidatos_municipais",
              "ids_aceitos", "ids_aceitos_municipais",
              "n_conhecidos", "n_conhecidos_nos_candidatos", "n_conhecidos_aceitos", "n_conhecidos_indisponiveis",
              "n_municipais_conhecidos", "n_municipais_conhecidos_reencontrados",
              "n_comentarios_janela", "status_comentarios"]
CAMPOS_VAR = ["alert_id", "variante", "consulta", "coletado_em", "status", "erro", "n_candidatos",
              "n_conhecidos", "n_conhecidos_nos_candidatos", "n_municipais_conhecidos", "n_municipais_conhecidos_reencontrados"]
CAMPOS_VID = ["alert_id", "variante", "coletado_em", "video_id", "aceito", "posicao", "search_query", "title",
              "channel_id", "channel_title", "published_at", "description_snippet", "description", "tags",
              "view_count", "comment_count", "relevance_score", "geo_score", "event_score", "geo_match_type",
              "municipal_evidence", "relevance_reasons", "rejection_reason", "filter_version"]
CAMPOS_COM = ["alert_id", "video_id", "comment_id", "parent_id", "is_reply", "published_at",
              "hours_from_alert_open", "like_count", "text_clean"]  # sem autor; menções já pseudonimizadas

def _anexar(arq, campos, linhas):
    if arq.exists():
        with open(arq, encoding="utf-8") as fh:
            cab = next(csv.reader(fh), [])
        if cab != campos:   # arquivo do lote 1 com colunas antigas: regrava com o cabeçalho novo
            antigos = _ler(arq)
            with open(arq, "w", encoding="utf-8", newline="") as fh:
                w = csv.DictWriter(fh, fieldnames=campos, extrasaction="ignore")
                w.writeheader(); w.writerows(antigos)
    novo = not arq.exists()
    with open(arq, "a", encoding="utf-8", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=campos, extrasaction="ignore")
        if novo:
            w.writeheader()
        w.writerows(linhas)

class Adiar(Exception):
    """Limite do dia atingido no meio do alerta: refaz na próxima execução."""

def _alerta(row):
    return {"cod_alerta": row["cod_alerta"], "municipio": row["municipio"], "uf": row["uf"],
            "codibge": row["codibge"], "evento": row["evento"], "nivel": row["nivel"], "datahoracriacao": row["opened_at"]}

def _consultas(alert, variante):
    if variante is None:
        return youtube.build_queries(alert)
    fam = next((t for f, t in YT_SEARCH_TERMS.items() if f != "default" and f in alert["evento"]), YT_SEARCH_TERMS["default"])
    mun = (alert["municipio"] or "").title()
    return [{"sem_uf": f"{fam[0]} {mun}", "chuva": f"chuva {mun}", "aspas": f'"{mun}" {fam[0]}'}[variante]]

_ultima_busca = [0.0]
INTERVALO_BUSCA = 0.8   # s entre buscas: o limite é 100 por minuto ("Search Queries per minute")

def _uma_busca(q, after, before, quota):
    espera = INTERVALO_BUSCA - (time.time() - _ultima_busca[0])
    if espera > 0:
        time.sleep(espera)
    r = youtube.search(q, after, before, KEY, CFG["MAX_VIDEOS_PER_ALERT"])
    _ultima_busca[0] = time.time(); quota.charge_search()
    return r

def _buscar(alert, queries, quota):
    after, before, _ = youtube.search_window(alert, CFG)
    seen = {}
    for q in queries:
        if not quota.can_search(True):
            raise Adiar()
        r = _uma_busca(q, after, before, quota)
        if r.status == "api_error" and r.error.startswith("429") and quota.can_search(True):
            # v2.1: limite POR MINUTO (não o diário) — espera e tenta uma vez
            print("  limite por minuto da busca: aguardando 65 s…")
            time.sleep(65)
            r = _uma_busca(q, after, before, quota)
        if r.status == "quota_exhausted":
            quota.search_exhausted = True
            raise Adiar()
        if r.status != "ok":
            return after, before, None, r.error[:200]
        for v in r.items:
            seen.setdefault(v["video_id"], {**v, "search_query": q})
    return after, before, seen, ""

def _stats(ids, quota):
    out = {}
    for i in range(0, len(ids), 50):
        if not quota.can_spend(1):
            raise Adiar()
        s = youtube.stats(ids[i:i + 50], KEY); quota.charge(1)
        if s.status == "quota_exhausted":
            quota.exhausted = True
            raise Adiar()
        out.update({x["video_id"]: x for x in s.items})
    return out

def _pontuar(row, alert, cands, stats, variante, agora):
    scored = []
    for v in cands:
        st = stats.get(v["video_id"], {})
        sc = score_video(v, alert, CFG)
        ev = municipal_evidence(row["municipio"], row["uf"], title=v.get("title", ""),
                                description=st.get("description", "") or v.get("description_snippet", ""))
        scored.append((v, sc, ev, st))
    aceitos = [t for t in scored if t[1]["is_relevant"]]
    aceitos.sort(key=lambda t: (is_municipal(t[2]), t[1]["geo_match_type"] == "municipality_direct", t[1]["relevance_score"],
                                int(t[3].get("comment_count", 0) or 0), int(t[3].get("view_count", 0) or 0)), reverse=True)
    ids_aceitos = [t[0]["video_id"] for t in aceitos[:CFG["MAX_VIDEOS_PER_ALERT"]]]
    videos = [{"alert_id": row["alert_id"], "variante": variante or "padrao", "coletado_em": agora,
               "video_id": v["video_id"], "aceito": int(v["video_id"] in ids_aceitos),
               "posicao": ids_aceitos.index(v["video_id"]) + 1 if v["video_id"] in ids_aceitos else "",
               "search_query": v["search_query"], "title": v.get("title", ""), "channel_id": v.get("channel_id", ""),
               "channel_title": v.get("channel_title", ""), "published_at": v.get("published_at", ""),
               "description_snippet": v.get("description_snippet", ""), "description": st.get("description", ""),
               "tags": st.get("tags", ""), "view_count": st.get("view_count", ""), "comment_count": st.get("comment_count", ""),
               "municipal_evidence": ev, "filter_version": FILTER_VERSION,
               **{k: sc[k] for k in ("relevance_score", "geo_score", "event_score", "geo_match_type",
                                     "relevance_reasons", "rejection_reason")}}
              for v, sc, ev, st in scored]
    ids_mun = [vid for vid in ids_aceitos if any(t[0]["video_id"] == vid and is_municipal(t[2]) for t in scored)]
    return scored, ids_aceitos, ids_mun, videos

def coletar(row, quota):
    alert = _alerta(row); agora = datetime.now(timezone.utc).isoformat(timespec="seconds")
    queries = _consultas(alert, None)
    res = {k: row.get(k, "") for k in ("alert_id", "grupo", "amostra", "ordem", "municipio", "uf", "evento", "opened_at")}
    after, before, seen, erro = _buscar(alert, queries, quota)
    res.update(coletado_em=agora, codigo_pipeline=COMMIT, filter_version=FILTER_VERSION,
               busca_de=after, busca_ate=before, consultas=" | ".join(queries), status="ok" if seen is not None else "api_error", erro=erro)
    if seen is None:
        return res, [], []
    cands = list(seen.values())
    stats = _stats([v["video_id"] for v in cands], quota)
    scored, ids_aceitos, ids_mun, videos = _pontuar(row, alert, cands, stats, None, agora)
    res.update(n_candidatos=len(cands), n_aceitos=len(ids_aceitos), n_aceitos_municipais=len(ids_mun),
               n_candidatos_municipais=sum(1 for _, _, ev, _ in scored if is_municipal(ev)),
               ids_aceitos=";".join(ids_aceitos), ids_aceitos_municipais=";".join(ids_mun))

    conhecidos = [v for v in row["videos_conhecidos"].split(";") if v]
    mun_conhecidos = [v for v in row["videos_municipais_conhecidos"].split(";") if v]
    if conhecidos:
        faltam = [v for v in conhecidos if v not in seen]
        indisponiveis = len(set(faltam) - set(_stats(faltam, quota))) if faltam else 0
        res.update(n_conhecidos=len(conhecidos), n_conhecidos_nos_candidatos=len(conhecidos) - len(faltam),
                   n_conhecidos_aceitos=sum(1 for v in conhecidos if v in ids_aceitos),
                   n_conhecidos_indisponiveis=indisponiveis, n_municipais_conhecidos=len(mun_conhecidos),
                   n_municipais_conhecidos_reencontrados=sum(1 for v in mun_conhecidos if v in seen))

    comentarios = []
    if COLETAR_COMENTARIOS and row["grupo"] == "silencio" and ids_mun:
        opened = parse_dt(row["opened_at"])
        fim = opened + timedelta(hours=CFG["EVENT_WINDOW_HOURS"]) if opened else None
        status_c = "ok"
        for vid in ids_mun:
            c = youtube.comments(vid, KEY, max_pages=MAX_PAGINAS_COMENTARIOS, order="time", quota=quota, window_end=fim)
            if c.status != "ok":
                status_c = c.status
            canon, links, _ = youtube.comment_links(c.items, alert=alert, link_scope="municipality_direct",
                                                     window_hours=CFG["EVENT_WINDOW_HOURS"])
            horas = {l["comment_id"]: l["hours_from_alert_open"] for l in links}
            for x in canon:
                comentarios.append({"alert_id": row["alert_id"], **{k: x.get(k, "") for k in CAMPOS_COM if k in x},
                                    "hours_from_alert_open": horas.get(x["comment_id"], "")})
        res.update(n_comentarios_janela=len(comentarios), status_comentarios=status_c)
    return res, videos, comentarios

def coletar_variante(row, variante, quota):
    alert = _alerta(row); agora = datetime.now(timezone.utc).isoformat(timespec="seconds")
    queries = _consultas(alert, variante)
    _, _, seen, erro = _buscar(alert, queries, quota)
    res = {"alert_id": row["alert_id"], "variante": variante, "consulta": queries[0], "coletado_em": agora,
           "status": "ok" if seen is not None else "api_error", "erro": erro}
    if seen is None:
        return res, []
    cands = list(seen.values())
    stats = _stats([v["video_id"] for v in cands], quota)
    _, _, _, videos = _pontuar(row, alert, cands, stats, variante, agora)
    conhecidos = [v for v in row["videos_conhecidos"].split(";") if v]
    mun = [v for v in row["videos_municipais_conhecidos"].split(";") if v]
    res.update(n_candidatos=len(cands), n_conhecidos=len(conhecidos),
               n_conhecidos_nos_candidatos=sum(1 for v in conhecidos if v in seen),
               n_municipais_conhecidos=len(mun), n_municipais_conhecidos_reencontrados=sum(1 for v in mun if v in seen))
    return res, videos

quota = youtube.QuotaTracker(ceiling=MAX_UNIDADES_EXECUCAO, high_reserve=0,
                             search_ceiling=MAX_BUSCAS_EXECUCAO, search_high_reserve=0)
n_ok = 0
for row, variante in pendentes:
    try:
        if variante:
            res, videos = coletar_variante(row, variante, quota); comentarios = []
        else:
            res, videos, comentarios = coletar(row, quota)
    except Adiar:
        print(f"limite do dia atingido em {row['alert_id']}: rode de novo amanhã (retoma daqui)")
        break
    _anexar(ARQ_VID, CAMPOS_VID, videos)
    _anexar(ARQ_COM, CAMPOS_COM, comentarios)
    _anexar(ARQ_VAR if variante else ARQ_RES, CAMPOS_VAR if variante else CAMPOS_RES, [res])  # por último
    n_ok += 1
    print(f"{row['ordem']:>4} {row['alert_id']:18s} {row['municipio'][:22]:22s} {variante or row['grupo']:10s} "
          f"cand {res.get('n_candidatos', 0):>2} aceitos {res.get('n_aceitos', '-'):>2} "
          f"citam município {res.get('n_aceitos_municipais', '-'):>2} {res['status']}")
    time.sleep(0.2)
print(f"\n{n_ok} itens nesta execução | buscas {quota.searches}/{MAX_BUSCAS_EXECUCAO} | unidades {quota.used}/{MAX_UNIDADES_EXECUCAO}")

In [ ]:
# ===== Limpeza: comentários de coletas substituídas (idempotente) =====
# Quando um alerta é refeito (ex.: tipo de evento corrigido), os comentários da coleta
# antiga ficam órfãos. Só ficam os comentários de vídeos municipais da coleta mais recente.
validos = {}
for r in _ler(ARQ_RES):
    if r["status"] == "ok" and r.get("evento", "") == ev_universo.get(r["alert_id"], "")             and r["coletado_em"] >= validos.get(r["alert_id"], {}).get("coletado_em", ""):
        validos[r["alert_id"]] = r
coms = _ler(ARQ_COM)
manter = [c for c in coms if c["video_id"] in (validos.get(c["alert_id"], {}).get("ids_aceitos_municipais", "") or "").split(";")]
if coms and len(manter) < len(coms):
    obsoletos = [c for c in coms if c not in manter]
    _anexar(pasta / "comentarios_obsoletos_APAGAR.csv", CAMPOS_COM, obsoletos)
    with open(ARQ_COM, "w", encoding="utf-8", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=CAMPOS_COM, extrasaction="ignore"); w.writeheader(); w.writerows(manter)
    print(f"{len(obsoletos)} comentários de coletas substituídas movidos para comentarios_obsoletos_APAGAR.csv")
else:
    print("nenhum comentário obsoleto")

In [ ]:
# ===== Complemento: descrição/canal/tags dos vídeos do lote 1 (sem gastar buscas) =====
vids = _ler(ARQ_VID)
ja = {r["video_id"] for r in _ler(ARQ_COMPL)}
faltam = sorted({r["video_id"] for r in vids if not r.get("description") and not r.get("channel_id")} - ja)
if faltam:
    q2 = youtube.QuotaTracker(ceiling=500, high_reserve=0)
    linhas = []
    for i in range(0, len(faltam), 50):
        r = youtube._request("videos", {"part": "snippet,statistics", "id": ",".join(faltam[i:i + 50]), "key": KEY}); q2.charge(1)
        if r.status != "ok":
            print("parou:", r.status, r.error[:120]); break
        for it in r.items:
            sn = it.get("snippet", {})
            linhas.append({"video_id": it["id"], "channel_id": sn.get("channelId", ""), "description": sn.get("description", ""),
                           "tags": "|".join(sn.get("tags", []) or []), "published_at": sn.get("publishedAt", "")})
    _anexar(ARQ_COMPL, ["video_id", "channel_id", "description", "tags", "published_at"], linhas)
    print(f"complemento: {len(linhas)} de {len(faltam)} vídeos ({q2.used} unidades); os que faltam foram apagados ou ficaram privados")
else:
    print("nada a complementar")

In [ ]:
# ===== Resumo =====
import math
from collections import Counter

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"),) * 3
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def _ultimo(linhas, chave):
    out = {}
    for r in linhas:
        if r["status"] == "ok" and r["coletado_em"] >= out.get(chave(r), {}).get("coletado_em", ""):
            out[chave(r)] = r
    return list(out.values())

res = [r for r in _ultimo(_ler(ARQ_RES), lambda r: r["alert_id"]) if r.get("evento", "") == ev_universo.get(r["alert_id"], "")]
cal = [r for r in res if r["grupo"] == "calibracao"]
if cal:
    tot = sum(int(r["n_conhecidos"] or 0) for r in cal); achou = sum(int(r["n_conhecidos_nos_candidatos"] or 0) for r in cal)
    mt = sum(int(r["n_municipais_conhecidos"] or 0) for r in cal); ma = sum(int(r["n_municipais_conhecidos_reencontrados"] or 0) for r in cal)
    al = [r for r in cal if int(r["n_municipais_conhecidos"] or 0) > 0]
    k = sum(1 for r in al if int(r["n_candidatos_municipais"] or 0) > 0)
    p, lo, hi = wilson(k, len(al))
    print(f"CALIBRAÇÃO ({len(cal)} alertas)")
    print(f"  conhecidos reencontrados pela busca: {achou}/{tot}; municipais: {ma}/{mt}")
    print(f"  SENSIBILIDADE DA BUSCA por alerta (≥1 candidato que cita o município): {k}/{len(al)} = {p:.0%} [IC95% {lo:.0%}–{hi:.0%}]")
var = _ultimo(_ler(ARQ_VAR), lambda r: (r["alert_id"], r["variante"]))
if var:
    print("\nVARIANTES DE CONSULTA (recuperação dos vídeos municipais conhecidos)")
    for v in ("sem_uf", "chuva", "aspas"):
        rs = [r for r in var if r["variante"] == v]
        if rs:
            print(f"  {v:7s}: {sum(int(r['n_municipais_conhecidos_reencontrados'] or 0) for r in rs)}/"
                  f"{sum(int(r['n_municipais_conhecidos'] or 0) for r in rs)} em {len(rs)} alertas")
am = [r for r in res if r["grupo"] == "silencio" and r["amostra"] == "1"]
if am:
    k = sum(1 for r in am if int(r["n_candidatos_municipais"] or 0) > 0)
    ka = sum(1 for r in am if int(r["n_aceitos_municipais"] or 0) > 0)
    p, lo, hi = wilson(ka, len(am))
    print(f"\nAMOSTRA DE SILÊNCIO ({len(am)} de 120)")
    print(f"  com ≥1 candidato que cita o município (a confirmar na anotação): {k}/{len(am)}")
    print(f"  aceitos pelos filtros ATUAIS (não calibrados): {ka}/{len(am)} = {p:.0%} [IC95% {lo:.0%}–{hi:.0%}]")
    print("  Não use esses números como resultado: a taxa definitiva sai da anotação humana (estudo de calibração).")

## Depois da coleta

- Mande para o Claude `resultados_alertas.csv`, `resultados_variantes.csv`, `videos.csv` e `videos_complemento.csv`.
  - Com eles, o script `scripts/build_annotation_sample.py` monta a planilha de anotação estratificada.
  - Os títulos e descrições são públicos, mas ficam fora do repositório.
- `comentarios.csv` **fica no Drive**.
- Apague `comentarios_lote1_ORIGINAL_apagar.csv` depois de conferir a pseudonimização.

**Referência.** Efstratiou, A. (2025). *I'm sorry Dave, I'm afraid I can't return that: On YouTube Search API use in research* (arXiv:2506.04422). https://arxiv.org/abs/2506.04422